In [9]:
! pip install -q rank-bm25 sentence-transformers faiss-cpu

In [10]:
import json
import pickle
import numpy as np
import faiss

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# --- Load the exact same artifacts produced in Level 1 ---

with open(r"C:\Users\HP\Downloads\safwa\recipe-rag-project\chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)


embeddings = np.load(r"C:\Users\HP\Downloads\safwa\recipe-rag-project\embeddings.npy")
DIM = embeddings.shape[1]
index = faiss.read_index(r"C:\Users\HP\Downloads\safwa\recipe-rag-project\recipe_index.faiss")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(f"Loaded {len(chunks)} chunks")
print(f"FAISS index: {index.ntotal} vectors, dim={DIM}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 69 chunks
FAISS index: 69 vectors, dim=384


In [11]:
# --- Build the sparse (BM25) index over the same chunks ---

def tokenize(text: str) -> list[str]:

    return text.lower().split()

corpus_tokens = [tokenize(c["text"]) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)  

print(f"BM25 index built over {len(corpus_tokens)} chunks")

BM25 index built over 69 chunks


In [14]:
# --- Dense retrieval ---

def dense_search(query: str, top_k: int = 5) -> list[dict]:
    """
    Semantic search using the FAISS index (same as Level 1's retrieve()).
    Returns a list of dicts: index, text, metadata, score (L2 distance,
    lower = more similar).
    """
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(query_vec, top_k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
        if idx == -1:
            continue
        chunk = chunks[idx]
        results.append({
            "rank": rank,
            "index": int(idx),
            "text": chunk["text"],
            "metadata": chunk["metadata"],
            "score": float(dist),
            "score_type": "L2 distance (lower = better)",
        })
    return results


# --- Sparse retrieval ---

def sparse_search(query: str, top_k: int = 5) -> list[dict]:
    """
    Keyword search using BM25 over the same chunk texts.
    Returns a list of dicts: index, text, metadata, score (BM25 score,
    higher = more similar).
    """
    scores = bm25.get_scores(tokenize(query))
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, 1):
        chunk = chunks[idx]
        results.append({
            "rank": rank,
            "index": int(idx),
            "text": chunk["text"],
            "metadata": chunk["metadata"],
            "score": float(scores[idx]),
            "score_type": "BM25 score (higher = better)",
        })
    return results

In [15]:
def reciprocal_rank_fusion(dense_results: list[dict], sparse_results: list[dict],
                            k: int = 60, top_n: int = 5) -> list[dict]:
    """
    Fuse dense + sparse ranked lists into one ranked list using RRF.
    Returns the fused, re-ranked list of chunk dicts (top_n).
    """
    rrf_scores = {}
    chunk_lookup = {}

    for res_list in (dense_results, sparse_results):
        for r in res_list:
            idx = r["index"]
            chunk_lookup[idx] = r
            rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (k + r["rank"])

    ranked = sorted(rrf_scores.items(), key=lambda kv: kv[1], reverse=True)[:top_n]

    fused = []
    for new_rank, (idx, score) in enumerate(ranked, 1):
        base = chunk_lookup[idx]
        fused.append({
            "rank": new_rank,
            "index": idx,
            "text": base["text"],
            "metadata": base["metadata"],
            "score": score,
            "score_type": "RRF score (higher = better)",
        })
    return fused


def hybrid_search(query: str, top_k: int = 5, candidate_k: int = 15,
                   rrf_k: int = 60) -> list[dict]:
    """
    Full hybrid retrieval: run dense + sparse search over a wider
    candidate pool, then fuse with RRF down to top_k.
    """
    dense_results = dense_search(query, top_k=candidate_k)
    sparse_results = sparse_search(query, top_k=candidate_k)
    return reciprocal_rank_fusion(dense_results, sparse_results, k=rrf_k, top_n=top_k)

In [16]:
def print_results(query: str, results: list[dict], label: str) -> None:
    print(f"\n {label} — \"{query}\" ")
    if not results:
        print("  (no results)")
        return
    for r in results:
        meta = r["metadata"]
        print(f"  #{r['rank']}  [{r['score_type']}: {r['score']:.4f}]  "
              f"{meta.get('title', 'Untitled')}  ({meta.get('url', '')})")
        print(f"       {r['text'].strip()[:150]}...")

In [17]:
comparison_queries = [
    "chicken tikka masala",                        
    "quick and easy egg breakfast",                
    "vegetarian dinner with soy sauce and ginger",  
]

comparison_log = []

for q in comparison_queries:
    dense_res = dense_search(q, top_k=5)
    hybrid_res = hybrid_search(q, top_k=5)

    print_results(q, dense_res, "DENSE-ONLY")
    print_results(q, hybrid_res, "HYBRID (RRF)")

    dense_titles = [r["metadata"].get("title") for r in dense_res]
    hybrid_titles = [r["metadata"].get("title") for r in hybrid_res]

    comparison_log.append({
        "query": q,
        "dense_top5_titles": dense_titles,
        "hybrid_top5_titles": hybrid_titles,
    })
    print("\n" + "=" * 70)


 DENSE-ONLY — "chicken tikka masala" 
  #1  [L2 distance (lower = better): 1.3006]  Thai Red Curry Soup  (https://www.allrecipes.com/recipe/274422/thai-red-curry-soup/)
       Recipe: Thai Red Curry Soup
Ingredients:
  - 2 tablespoons olive oil, divided
  - 18 large shrimp, peeled and deveined
  - ¼ teaspoon kosher salt
  - ...
  #2  [L2 distance (lower = better): 1.3718]  The Cheesecake Factory Santa Fe Salad  (https://www.allrecipes.com/recipe/258165/the-cheesecake-factory-santa-fe-salad/)
       Recipe: The Cheesecake Factory Santa Fe Salad
Ingredients:
  - 4 skinless, boneless chicken breasts
  - ½ cup teriyaki marinade
  - ¼ cup chopped fres...
  #3  [L2 distance (lower = better): 1.4011]  My Favorite No-Mayo Egg Salad  (https://www.allrecipes.com/recipe/236330/my-favorite-no-mayo-egg-salad/)
       Recipe: My Favorite No-Mayo Egg Salad
Ingredients:
  - 1 tablespoon plain yogurt
  - 2 teaspoons tahini
  - 2 teaspoons za'atar
  - salt and ground bl...
  #4  [L2 distance (lower = b